# 데모 백엔드를 Colab GPU에서 띄우기

런타임 → GPU(T4)로 바꾸고 위에서부터 실행한다. 마지막 셀이 `https://….trycloudflare.com` 주소를 출력하면 그 주소가 데모 화면이다 (프론트도 같은 서버가 내보낸다).
런타임이 끊기면 주소도 사라지므로, 다시 실행하면 새 주소가 나온다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U "transformers>=5,<6" fastapi uvicorn

ROOT = '/content/drive/MyDrive/fact-verification'
!rm -rf /content/fv && git clone -q --depth 1 https://github.com/kohjunseo1/Fact-Verfication.git /content/fv
DEMO = '/content/fv/한기선/demo'

In [ ]:
# 위키 DB와 FTS5 인덱스는 Drive에서 바로 읽으면 느려서 Colab 로컬 디스크로 복사한다.
# 인덱스는 처음 한 번 여기서 만들고(약 1~2분) Drive에 저장해 둔다
import os
WIKI, INDEX = '/content/wiki_wo_links.db', '/content/wiki_index.db'
if not os.path.exists(WIKI):
    !cp "{ROOT}/data/raw/wiki_wo_links.db" {WIKI}
if not os.path.exists(INDEX):
    if os.path.exists(f'{ROOT}/data/index/wiki_index.db'):
        !cp "{ROOT}/data/index/wiki_index.db" {INDEX}
    else:
        !python3 "{DEMO}/build_index.py" {WIKI} {INDEX}
        !mkdir -p "{ROOT}/data/index" && cp {INDEX} "{ROOT}/data/index/wiki_index.db"

In [ ]:
# app.py(/predict, /health, /info)를 /api 아래에, frontend/index.html을 / 에 붙인다. k8s의 nginx가 하던 일이다
open(f'{DEMO}/backend/colab_serve.py', 'w').write('''
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles
import app
root = FastAPI()
root.mount('/api', app.app)
root.mount('/', StaticFiles(directory='../frontend', html=True))
''')

import subprocess, time, urllib.request
env = {**os.environ, 'WIKI_DB': WIKI, 'INDEX_DB': INDEX, 'MODEL_DIR': f'{ROOT}/models'}
server = subprocess.Popen('uvicorn colab_serve:root --port 8000 > /content/server.log 2>&1', shell=True,
                          cwd=f'{DEMO}/backend', env=env)
for _ in range(120):  # 모델 4개를 GPU에 올리는 동안 기다린다
    try:
        urllib.request.urlopen('http://localhost:8000/api/health'); print('서버 준비됨'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/content/server.log').read()[-3000:])

In [ ]:
# 계정 없이 쓰는 cloudflared 임시 터널로 바깥에 연다
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
tunnel = subprocess.Popen('/content/cloudflared tunnel --url http://localhost:8000 > /content/tunnel.log 2>&1', shell=True)
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        print('데모 주소:', m.group()); break

In [ ]:
# 동작 확인 (GPU면 몇 초 안에 끝난다)
import json
req = urllib.request.Request('http://localhost:8000/api/predict', method='POST', headers={'Content-Type': 'application/json'},
                             data=json.dumps({'claim': 'The Assignation was written by a playwright who became Poet Laureate in 1668.'}).encode())
r = json.load(urllib.request.urlopen(req))
print(r['label'], r['prob_supported'], r['timing'], r['latency_ms'], 'ms')